# SofaScore — extrair coordenadas da partida

Cole a **URL da partida** na célula de configuração e execute todas as células.

O notebook gera um **CSV único** com passes e conduções de **todos os jogadores**, incluindo:
- coordenadas (`x`, `y`, `end_x`, `end_y`)
- jogador, time, posição
- tempo do evento e demais campos retornados pela API

> Se aparecer erro **403**, altere `MODE = "browser"` (precisa do Google Chrome instalado).

## Como exportar `lineups.json` (se a API der 403)

1. Abra a partida no **Google Chrome**
2. Pressione **F12** → aba **Network** (Rede)
3. Recarregue a página (**F5**)
4. No filtro, digite: `lineups`
5. Clique na requisição que termina com `/lineups` (status **200**)
6. Aba **Response** (Resposta)
7. Clique com o botão direito no JSON → **Copy response** (Copiar resposta)
8. Cole no Bloco de Notas e salve como `lineups.json` em `C:\Users\Walt\Desktop\Sofascore\`

O arquivo deve começar assim:

```json
{
  "home": {
    "players": [
      {
        "player": {
          "id": 12994,
          "name": "Lionel Messi"
        }
      }
    ]
  },
  "away": { ... }
}
```

**Não** salve a página inteira (Ctrl+S). **Não** copie da aba Preview se vier vazia.

Depois use no notebook:

```python
lineups_file=PROJECT_ROOT / "lineups.json"
```

## Passo 2 — obter `rating-breakdown` (coordenadas)

O `lineups.json` só lista jogadores. As coordenadas vêm de `rating-breakdown`.

**Importante:** colar a URL da API numa aba nova → quase sempre **403 challenge**.  
Isso é normal. O SofaScore exige sessão da **página da partida**.

### Método recomendado — exportar HAR (1 vez, todos os jogadores)

1. Abra a partida no Chrome (página normal, não URL da API)
2. **F12** → **Network** → marque **Preserve log**
3. Clique em **cada jogador** na escalação (um por um)
4. Botão direito na lista de requisições → **Save all as HAR with content**
5. Salve como `partida.har` na pasta do projeto
6. Execute a célula **"Extrair do HAR"** abaixo

### Método alternativo — 1 jogador (teste)

1. Abra a partida → clique em **Messi**
2. Network → filtre `rating-breakdown` → **Copy response**
3. Salve em `notebook_output/15186502/raw/12994.json`

In [ ]:
import sys
from pathlib import Path

# Garante que o Python encontre extract_sofascore_match_events.py
# mesmo se o Jupyter iniciar em outra pasta.


def find_project_root(marker: str = "extract_sofascore_match_events.py") -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for base in candidates:
        if (base / marker).exists():
            return base.resolve()
    raise FileNotFoundError(
        "Não encontrei extract_sofascore_match_events.py.\n"
        "Abra o notebook na pasta do repositório (onde está o .py) "
        "ou faça git pull da branch cursor/sofascore-extract-events-88ee."
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Pasta do projeto: {PROJECT_ROOT}")

# Execute uma vez se faltar alguma dependência:
# %pip install -q curl_cffi pandas undetected-chromedriver selenium beautifulsoup4 ipykernel

In [ ]:
⬇️ Configure `MATCH_URL` na próxima célula, depois execute o download e a extração.

In [ ]:
import pandas as pd

from extract_sofascore_match_events import extract_match_events, parse_match_id

# ========== CONFIGURAÇÃO — edite aqui ==========
MATCH_URL = "https://www.sofascore.com/football/match/argentina-austria/tUbsuWb#id:15186502"

# "auto" tenta curl -> Chrome headless -> Chrome visível
# Se já recebeu 403, use direto:
MODE = "browser"

# nome do CSV final (passes + conduções de todos os jogadores)
OUTPUT_CSV = PROJECT_ROOT / "coordenadas_partida.csv"
# ===============================================

match_id = parse_match_id(MATCH_URL)
output_dir = PROJECT_ROOT / "notebook_output" / str(match_id)
output_dir

In [ ]:
from extract_sofascore_match_events import download_rating_breakdowns

RAW_DIR = output_dir / "raw"
LINEUPS_FILE = PROJECT_ROOT / "lineups.json"

# Abre o Chrome visível e baixa um JSON por jogador em RAW_DIR
download_rating_breakdowns(
    MATCH_URL,
    LINEUPS_FILE,
    output_dir,
    mode="browser",
    delay=1.5,
    base_dir=PROJECT_ROOT,
)

In [ ]:
# --- MÉTODO HAR (recomendado) ---
# Depois de gravar partida.har clicando em cada jogador na escalação:
from extract_sofascore_match_events import parse_har_rating_breakdowns

HAR_FILE = PROJECT_ROOT / "partida.har"  # ajuste o caminho se necessário
RAW_DIR = parse_har_rating_breakdowns(HAR_FILE, output_dir, match_id=match_id)
print("Pronto:", RAW_DIR)

In [ ]:
from extract_sofascore_match_events import extract_match_events

result = extract_match_events(
    MATCH_URL,
    output_dir,
    mode="browser",
    lineups_file=PROJECT_ROOT / "lineups.json",
    rating_dir=RAW_DIR,
)

df = result["all_events"].copy()
print(f"Jogadores no lineup: {len(result['lineups'])}")
print(f"Passes: {len(result['passes'])} | Conduções: {len(result['ball_carries'])}")
print(f"Total de eventos: {len(df)}")
df.head()

In [ ]:
# Colunas principais de coordenadas (o CSV salva todas as colunas disponíveis)
coord_cols = [
    "match_id", "category", "player_name", "team", "side", "player_position",
    "x", "y", "end_x", "end_y", "time", "timeSeconds", "outcome", "length", "progressive",
]
preview_cols = [c for c in coord_cols if c in df.columns]
df[preview_cols].head(10)

In [ ]:
csv_path = Path(OUTPUT_CSV)
df.to_csv(csv_path, index=False)

print(f"CSV salvo em: {csv_path}")
print(f"Linhas: {len(df)} | Colunas: {len(df.columns)}")
df["category"].value_counts()